# Task 115: adapt_constitutive — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Constitutive model calibration for material behavior

**Paper**: No dedicated paper — standard constitutive model inverse calibration
**Repository**: https://github.com/aschowtjak/ADAPT

### Physical Background

Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.

### Forward Model

```
y = f(theta) + n
```

### Inverse Problem

```
theta_hat = argmin ||y - f(theta)||^2  or Bayesian inference
```

### Performance Metrics
- **PSNR**: 55.51 dB
- **SSIM**: N/A
- **Evaluation**: Constitutive parameter identification (CC=0.9999, E_RE=2.3%, σ_y_RE=3.7%)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
adapt_constitutive — Constitutive Model Calibration
=====================================================
From experimental stress-strain data, calibrate constitutive model parameters
(elastic modulus, yield stress, hardening coefficient, hardening exponent) by
inverse optimisation using a power-law hardening model.

Physics:
  Forward:
    σ = E × ε                               for ε ≤ ε_y  (elastic)
    σ = σ_y + K × (ε − ε_y)^n              for ε > ε_y  (plastic, power-law)
    where ε_y = σ_y / E

  Inverse:
    scipy.optimize.minimize (L-BFGS-B) minimises ||σ_obs − σ_model(params)||²
    to recover [E, σ_y, K, n] from a noisy stress-strain curve.
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`constitutive_model`, `objective`, `calibrate`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1.  FORWARD MODEL
# ═══════════════════════════════════════════════════════════════════
def constitutive_model(strain, E, sigma_y, K, n):
    """
    Power-law hardening constitutive model.

    σ = E ε                          for ε ≤ ε_y
    σ = σ_y + K (ε − ε_y)^n         for ε >  ε_y
    """
    eps_y = sigma_y / E
    stress = np.empty_like(strain)
    elastic = strain <= eps_y
    plastic = ~elastic
    stress[elastic] = E * strain[elastic]
    eps_p = strain[plastic] - eps_y
    eps_p = np.maximum(eps_p, 0.0)
    stress[plastic] = sigma_y + K * np.power(eps_p, n)
    return stress

# ═══════════════════════════════════════════════════════════════════
# 3.  INVERSE:  L-BFGS-B OPTIMISATION
# ═══════════════════════════════════════════════════════════════════
def objective(params, strain, stress_obs):
    """Sum-of-squares misfit."""
    E, sigma_y, K, n = params
    stress_pred = constitutive_model(strain, E, sigma_y, K, n)
    return np.sum((stress_obs - stress_pred) ** 2)

def calibrate(strain, stress_obs):
    """Recover constitutive parameters from noisy stress-strain data."""
    # Estimate E from initial slope (first ~10% of data where elastic)
    n_init = max(5, len(strain) // 20)
    E_est = float(np.polyfit(strain[:n_init], stress_obs[:n_init], 1)[0])
    E_est = np.clip(E_est, 50e3, 500e3)

    # Estimate yield stress from the knee
    sy_est = float(stress_obs[n_init * 2]) if n_init * 2 < len(stress_obs) else 300.0

    x0 = [E_est, sy_est, 700.0, 0.4]
    bounds = [(50e3, 500e3),   # E
              (100.0, 800.0),  # σ_y
              (100.0, 2000.0), # K
              (0.05, 0.95)]    # n

    best_result = None
    best_cost = np.inf
    # Multi-start to avoid local minima
    starts = [
        x0,
        [E_est * 0.9, sy_est * 1.1, 800.0, 0.5],
        [E_est * 1.1, sy_est * 0.9, 600.0, 0.35],
        [200e3, 350.0, 800.0, 0.45],
    ]
    for s in starts:
        result = minimize(objective, s, args=(strain, stress_obs),
                          method="L-BFGS-B", bounds=bounds,
                          options={"maxiter": 10000, "ftol": 1e-18, "gtol": 1e-14})
        if result.fun < best_cost:
            best_cost = result.fun
            best_result = result
    return best_result.x

# ═══════════════════════════════════════════════════════════════════
# 6.  MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("adapt_constitutive — Constitutive Model Calibration")
    print("=" * 60)

    # 1. Strain grid
    strain = np.linspace(0, STRAIN_MAX, N_POINTS)

    # 2. Ground truth
    print("[1/4] Generating ground-truth stress-strain curve ...")
    stress_gt, stress_noisy = generate_gt(strain)

    # 3. Inverse calibration
    print("[2/4] Running L-BFGS-B inverse calibration ...")
    params_fit = calibrate(strain, stress_noisy)
    print(f"  Fitted: E={params_fit[0]:.1f}, σ_y={params_fit[1]:.2f}, "
          f"K={params_fit[2]:.2f}, n={params_fit[3]:.4f}")

    # 4. Reconstruct stress curve with fitted params
    stress_recon = constitutive_model(strain, *params_fit)

    # 5. Metrics
    params_true = [E_TRUE, SY_TRUE, K_TRUE, N_TRUE]
    metrics = compute_metrics(stress_gt, stress_recon, params_true, params_fit)
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  CC   = {metrics['CC']:.6f}")
    print(f"  RMSE = {metrics['RMSE']:.4f} MPa")
    for k in ["E", "sigma_y", "K", "n"]:
        print(f"  {k}: true={metrics[f'{k}_true']:.4f}  fit={metrics[f'{k}_fitted']:.4f}  RE={metrics[f'{k}_RE_pct']:.2f}%")

    # 6. Save
    print("[3/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), stress_gt)
        np.save(os.path.join(d, "recon_output.npy"), stress_recon)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # 7. Visualise
    print("[4/4] Generating visualisation ...")
    visualize(strain, stress_gt, stress_noisy, stress_recon, params_true, params_fit, metrics)

    print("Done ✓")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = f(theta) + n
```

Functions: `generate_gt`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 2.  GENERATE GROUND-TRUTH (synthetic experiment)
# ═══════════════════════════════════════════════════════════════════
def generate_gt(strain):
    """Return clean + noisy stress for the GT parameter set."""
    stress_clean = constitutive_model(strain, E_TRUE, SY_TRUE, K_TRUE, N_TRUE)
    noise = NOISE_LEVEL * np.max(np.abs(stress_clean)) * np.random.randn(len(strain))
    stress_noisy = stress_clean + noise
    return stress_clean, stress_noisy

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4.  METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(stress_gt, stress_recon, params_true, params_fit):
    """PSNR, CC, parameter relative errors."""
    # PSNR on stress curve
    mse = np.mean((stress_gt - stress_recon) ** 2)
    data_range = np.max(stress_gt) - np.min(stress_gt)
    psnr = 10.0 * np.log10(data_range ** 2 / (mse + 1e-30))

    # Correlation coefficient
    cc = float(np.corrcoef(stress_gt.ravel(), stress_recon.ravel())[0, 1])

    # RMSE
    rmse = float(np.sqrt(mse))

    # Parameter relative errors
    names = ["E", "sigma_y", "K", "n"]
    param_errors = {}
    for name, pt, pf in zip(names, params_true, params_fit):
        re = abs(pf - pt) / abs(pt) * 100.0
        param_errors[f"{name}_true"] = float(pt)
        param_errors[f"{name}_fitted"] = float(pf)
        param_errors[f"{name}_RE_pct"] = float(re)

    metrics = {"PSNR": float(psnr), "CC": float(cc), "RMSE": float(rmse)}
    metrics.update(param_errors)
    return metrics

# ═══════════════════════════════════════════════════════════════════
# 5.  VISUALISATION
# ═══════════════════════════════════════════════════════════════════
def visualize(strain, stress_gt, stress_noisy, stress_recon, params_true, params_fit, metrics):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # (a) GT vs Fitted stress-strain curve
    ax = axes[0]
    ax.plot(strain * 100, stress_gt, "k-", lw=2, label="Ground Truth")
    ax.plot(strain * 100, stress_noisy, ".", color="gray", ms=1, alpha=0.4, label="Noisy Observation")
    ax.plot(strain * 100, stress_recon, "r--", lw=2, label="Fitted Model")
    ax.set_xlabel("Strain (%)")
    ax.set_ylabel("Stress (MPa)")
    ax.set_title(f"Stress-Strain Curve  (PSNR={metrics['PSNR']:.1f} dB)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (b) Parameter comparison
    ax = axes[1]
    names = ["E (MPa)", "σ_y (MPa)", "K (MPa)", "n"]
    keys  = ["E", "sigma_y", "K", "n"]
    true_vals = [metrics[f"{k}_true"] for k in keys]
    fit_vals  = [metrics[f"{k}_fitted"] for k in keys]
    # Normalise for bar chart
    norm_true = [1.0] * 4
    norm_fit  = [f / t if t != 0 else 0 for f, t in zip(fit_vals, true_vals)]
    x = np.arange(4)
    ax.bar(x - 0.18, norm_true, 0.35, label="True", color="steelblue")
    ax.bar(x + 0.18, norm_fit,  0.35, label="Fitted", color="salmon")
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=9)
    ax.set_ylabel("Normalised Value")
    ax.set_title("Parameter Comparison (normalised)")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    # (c) Residual
    ax = axes[2]
    residual = stress_gt - stress_recon
    ax.plot(strain * 100, residual, "b-", lw=1)
    ax.axhline(0, color="k", ls="--", lw=0.5)
    ax.set_xlabel("Strain (%)")
    ax.set_ylabel("Residual Stress (MPa)")
    ax.set_title(f"Residual  (RMSE={metrics['RMSE']:.2f} MPa, CC={metrics['CC']:.4f})")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("adapt_constitutive — Constitutive Model Calibration")
print("=" * 60)

### 1. Strain grid

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 1. Strain grid
strain = np.linspace(0, STRAIN_MAX, N_POINTS)

### 2. Ground truth

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 2. Ground truth
print("[1/4] Generating ground-truth stress-strain curve ...")
stress_gt, stress_noisy = generate_gt(strain)

### 3. Inverse calibration

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Inverse calibration
print("[2/4] Running L-BFGS-B inverse calibration ...")
params_fit = calibrate(strain, stress_noisy)
print(f"  Fitted: E={params_fit[0]:.1f}, σ_y={params_fit[1]:.2f}, "
      f"K={params_fit[2]:.2f}, n={params_fit[3]:.4f}")

### 4. Reconstruct stress curve with fitted params

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Reconstruct stress curve with fitted params
stress_recon = constitutive_model(strain, *params_fit)

### 5. Metrics

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 5. Metrics
params_true = [E_TRUE, SY_TRUE, K_TRUE, N_TRUE]
metrics = compute_metrics(stress_gt, stress_recon, params_true, params_fit)
print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  CC   = {metrics['CC']:.6f}")
print(f"  RMSE = {metrics['RMSE']:.4f} MPa")
for k in ["E", "sigma_y", "K", "n"]:
    print(f"  {k}: true={metrics[f'{k}_true']:.4f}  fit={metrics[f'{k}_fitted']:.4f}  RE={metrics[f'{k}_RE_pct']:.2f}%")

### 6. Save

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 6. Save
print("[3/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    np.save(os.path.join(d, "gt_output.npy"), stress_gt)
    np.save(os.path.join(d, "recon_output.npy"), stress_recon)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

### 7. Visualise

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 7. Visualise
print("[4/4] Generating visualisation ...")
visualize(strain, stress_gt, stress_noisy, stress_recon, params_true, params_fit, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **adapt_constitutive**:

1. **Problem Setup**: Physical systems have parameters (frequencies, damping, material constants) extracted by fitting models to measurements.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=55.51 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: No dedicated paper — standard constitutive model inverse calibration
- Repository: https://github.com/aschowtjak/ADAPT
